# 04. Data Cleaning, Missing Values & Type Hygiene: Beginner Guide

### 📌 Overview
Master **04. Data Cleaning, Missing Values & Type Hygiene: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Null Detection & Handling**: Covers `.isnull()`/`.isna()`, `.notnull()`/`.notna()`, `.dropna()`, and `.fillna()`.
- **Deduplication**: Covers `.duplicated()` and `.drop_duplicates()`.
- **Type Hygiene & Conversion**: Covers `.astype()`, `pd.to_numeric()`, and `pd.to_datetime()`.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Detecting Missing Values: `DataFrame.isna()` / `.isnull()`
- **What it does:** Detects missing or null values (`NaN`, `None`, `pd.NA`), returning a boolean mask of identical shape.
- **Syntax:** `DataFrame.isna()`
- **Key Note:** `.isna()` and `.isnull()` are exact aliases in Pandas. Combine with `.sum()` to compute per-column null counts.
- **Dataset Application & Code Demonstration:** Executes Detecting Missing Values on the transaction DataFrame (`df`) to inspect and transform tabular features.


In [2]:
print('Missing Values Count per Column:\n', df.isnull().sum())

Missing Values Count per Column:
 transaction_id          0
customer_id             0
merchant_id             0
transaction_amount    749
card_type               0
transaction_status      0
device_type             0
account_age_months      0
transaction_date        0
region                  0
is_fraud                0
dtype: int64


### 🔹 Detecting Valid Values: `DataFrame.notna()` / `.notnull()`
- **What it does:** Detects non-missing values, returning a boolean mask where True indicates valid, populated data.
- **Syntax:** `DataFrame.notna()`
- **Key Note:** Useful for building inverse filter masks to isolate fully populated records without generating null warnings.
- **Dataset Application & Code Demonstration:** Applies Detecting Valid Values on fintech records using columns `customer_id` to demonstrate real-world execution.


In [3]:
print('Valid Customer Rows Count:', df['customer_id'].notnull().sum())

Valid Customer Rows Count: 15000


### 🔹 Dropping Missing Values: `DataFrame.dropna()`
- **What it does:** Removes missing values by dropping entire rows or columns containing null entries.
- **Syntax:** `DataFrame.dropna()`
- **Key Note:** Always specify `subset=['critical_col']` rather than globally dropping rows across the whole dataframe to avoid unintended row loss.
- **Dataset Application & Code Demonstration:** Applies Dropping Missing Values on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [4]:
cleaned_df = df.dropna(subset=['transaction_amount'])
print('Rows after dropna on amount:', len(cleaned_df))

Rows after dropna on amount: 14251


### 🔹 Imputing Missing Values: `DataFrame.fillna()`
- **What it does:** Fills `NA`/`NaN` values using a specified scalar value, dictionary of column mappings, or interpolation strategy.
- **Syntax:** `DataFrame.fillna()`
- **Key Note:** Passing a dictionary mapping (`{'numeric_col': 0.0, 'cat_col': 'UNKNOWN'}`) allows clean per-column imputation in a single call.
- **Dataset Application & Code Demonstration:** Applies Imputing Missing Values on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [5]:
imputed_amt = df['transaction_amount'].fillna(df['transaction_amount'].median())
print('Null count after median imputation:', imputed_amt.isnull().sum())

Null count after median imputation: 0


### 🔹 Detecting Duplicates: `DataFrame.duplicated()`
- **What it does:** Identifies duplicate rows based on all columns or a specified subset, returning a boolean Series.
- **Syntax:** `DataFrame.duplicated()`
- **Key Note:** Setting `keep=False` isolates all duplicate occurrences including the first, ideal for investigating data pipeline duplication bugs.
- **Dataset Application & Code Demonstration:** Applies Detecting Duplicates on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [6]:
duplicate_count = df.duplicated(subset=['transaction_id']).sum()
print('Duplicate Transaction IDs Count:', duplicate_count)

Duplicate Transaction IDs Count: 100


### 🔹 Removing Duplicates: `DataFrame.drop_duplicates()`
- **What it does:** Removes duplicate rows from the DataFrame, retaining only unique occurrences based on specified key columns.
- **Syntax:** `DataFrame.drop_duplicates()`
- **Key Note:** Always verify sorting order before dropping duplicates with `keep='last'` to ensure the latest valid record is preserved.
- **Dataset Application & Code Demonstration:** Applies Removing Duplicates on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [7]:
deduped_df = df.drop_duplicates(subset=['transaction_id'], keep='first')
print(f'Original Rows: {len(df)} -> Deduplicated Rows: {len(deduped_df)}')

Original Rows: 15000 -> Deduplicated Rows: 14900


### 🔹 Type Casting: `DataFrame.astype()` / `Series.astype()`
- **What it does:** Casts a pandas object to a specified `dtype` (e.g., int32, float64, bool, category, string).
- **Syntax:** `DataFrame.astype()`
- **Key Note:** Passing a dictionary (`{'colA': 'int32', 'colB': 'category'}`) allows batch casting across multiple columns simultaneously.
- **Dataset Application & Code Demonstration:** Applies Type Casting on fintech records using columns `is_fraud`, `transaction_amount` to demonstrate real-world execution.


In [8]:
optimized_df = deduped_df.astype({'is_fraud': 'int8', 'transaction_amount': 'float32'})
print('Optimized Dtypes:\n', optimized_df.dtypes[['transaction_amount', 'is_fraud']])

Optimized Dtypes:
 transaction_amount    float32
is_fraud                 int8
dtype: object


### 🔹 Robust Numeric Conversion: `pd.to_numeric()`
- **What it does:** Converts an argument to a numeric type, safely coercing invalid/unparseable strings to `NaN` or downcasting to the smallest possible integer/float.
- **Syntax:** `pd.to_numeric()`
- **Key Note:** Using `errors='coerce'` safely handles dirty text data containing rogue characters (e.g. '$', 'N/A') by converting them to `NaN` without crashing.
- **Dataset Application & Code Demonstration:** Applies Robust Numeric Conversion on fintech records using columns `transaction_amount` to demonstrate real-world execution.


In [9]:
coerced_nums = pd.to_numeric(df['transaction_amount'], errors='coerce')
print('Parsed Numeric Count:', coerced_nums.count())

Parsed Numeric Count: 14251


### 🔹 Robust Datetime Conversion: `pd.to_datetime()`
- **What it does:** Converts scalar, array-like, or Series strings/timestamps into standard pandas `Timestamp` or `DatetimeIndex`.
- **Syntax:** `pd.to_datetime()`
- **Key Note:** Supplying an explicit `format` string avoids heuristic guessing and accelerates datetime parsing speed by up to 20x on large datasets.
- **Dataset Application & Code Demonstration:** Applies Robust Datetime Conversion on fintech records using columns `transaction_date` to demonstrate real-world execution.


In [10]:
clean_dates = pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce')
print('Parsed Datetime Series Head:\n', clean_dates.head())

Parsed Datetime Series Head:
 0   2026-02-17 08:28:57
1   2025-01-03 00:00:00
2   2025-12-20 00:00:00
3   2026-02-06 03:39:02
4   2025-01-07 00:23:37
Name: transaction_date, dtype: datetime64[ns]


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: End-to-End Data Cleaning Pipeline
- **Objective:** Q1: End-to-End Data Cleaning Pipeline
- **Approach:** Build an end-to-end cleaning pipeline on raw_transactions.csv.
- **Syntax:** `df.drop_duplicates().dropna().assign(...)`

In [11]:
pipeline_df = (
    df
    .drop_duplicates(subset=['transaction_id'])
    .assign(
        transaction_date=lambda d: pd.to_datetime(d['transaction_date'], format='mixed', errors='coerce'),
        transaction_amount=lambda d: pd.to_numeric(d['transaction_amount'], errors='coerce')
    )
    .dropna(subset=['transaction_amount', 'transaction_date'])
)
print(f'Cleaned Dataset Ready for Modeling: {len(pipeline_df)} valid transactions')

Cleaned Dataset Ready for Modeling: 14157 valid transactions